# IBM Quantum Workshop
1. Set your options in the **CONFIG** cell.
2. Run every cell in order (Shift+Enter).
3. In the last cell, un-comment the example you want and run it.

In [ ]:
# ============ CONFIG ============
USE_SIMULATOR = True   # True = run locally, False = run on your real IBM Quantum instance

# Only needed if USE_SIMULATOR = False:
API_KEY = ""           # 44-character API key from your IBM Quantum dashboard
CRN = ""               # starts with crn:v1:bluemix... (Instances page)

In [ ]:
# ============ SETUP (just run this) ============
import numpy as np
from fractions import Fraction
from math import gcd
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.transpiler import generate_preset_pass_manager

BACKEND = None

if USE_SIMULATOR:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import SamplerV2 as Sampler
    BACKEND = AerSimulator()
    print('Ready: local simulator')
else:
    # --- verify credentials before connecting ---
    problems = []
    if not API_KEY.strip():
        problems.append('API_KEY is empty. Paste it into the CONFIG cell above.')
    elif len(API_KEY.strip()) != 44:
        problems.append(f'API_KEY should be 44 characters (yours is {len(API_KEY.strip())}). Copy it again from the dashboard.')
    if not CRN.strip():
        problems.append('CRN is empty. Paste it into the CONFIG cell above.')
    elif not CRN.strip().startswith('crn:'):
        problems.append('CRN should start with "crn:". Copy it from the Instances page (hover the CRN, click copy).')
    if problems:
        for p in problems: print('PROBLEM:', p)
        raise RuntimeError('Fix the CONFIG cell, run it, then run this cell again.')

    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    try:
        service = QiskitRuntimeService(
            channel='ibm_quantum_platform',
            token=API_KEY.strip(),
            instance=CRN.strip(),
        )
        BACKEND = service.least_busy(operational=True, simulator=False)
    except Exception as e:
        msg = str(e)
        print('Could not connect. The error was:\n ', msg, '\n')
        low = msg.lower()
        if '401' in low or 'unauthorized' in low or 'invalid' in low and 'token' in low:
            print('HINT: Your API key looks wrong or was revoked. Create a new one on the dashboard.')
        elif 'crn' in low or 'instance' in low or 'not found' in low or '404' in low:
            print('HINT: Your CRN looks wrong, or the instance is in a different region.')
            print('      Open instances live in us-east: check the region switcher in the site header.')
        elif 'connection' in low or 'timeout' in low or 'resolve' in low:
            print('HINT: Network problem reaching IBM Cloud. Check your internet and retry.')
        else:
            print('HINT: Double-check both values in CONFIG. If it persists, create a fresh API key.')
        raise RuntimeError('Fix the issue above, then re-run this cell.')
    print('Connected. Running on real hardware:', BACKEND.name, f'({BACKEND.num_qubits} qubits)')

def run_circuit(qc, shots=1024):
    """Transpile for the chosen backend, run, return counts."""
    if USE_SIMULATOR:
        tqc = transpile(qc, BACKEND)
        result = Sampler().run([tqc], shots=shots).result()
    else:
        pm = generate_preset_pass_manager(optimization_level=1, backend=BACKEND)
        tqc = pm.run(qc)
        result = Sampler(mode=BACKEND).run([tqc], shots=shots).result()
    return result[0].data.meas.get_counts()

In [ ]:
# ============ EXAMPLES (just run this to define them) ============

def example_1():
    """Hello quantum: 5 coin flips. One qubit, 50/50 superposition, measured."""
    # Note: on real hardware each loop iteration is a separate queued job.
    for i in range(5):
        qc = QuantumCircuit(QuantumRegister(1), ClassicalRegister(1, 'meas'))
        qc.h(0)
        qc.measure(0, 0)
        counts = run_circuit(qc, shots=1)
        print(f'Run {i+1}: measured {list(counts)[0]}')

def example_2():
    """Bell state: two entangled qubits. Results are (almost) only 00 and 11."""
    qc = QuantumCircuit(QuantumRegister(2), ClassicalRegister(2, 'meas'))
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    counts = run_circuit(qc, shots=1024)
    print('Counts:', counts)

def example_3(N=15, a=7):
    """Shor's algorithm: factor 15 using quantum period finding."""
    def c_amod15(a, power):
        U = QuantumCircuit(4)
        for _ in range(power):
            if a in [2, 13]: U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
            if a in [7, 8]:  U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
            if a in [4, 11]: U.swap(1, 3); U.swap(0, 2)
            if a in [7, 11, 13]:
                for q in range(4): U.x(q)
        g = U.to_gate(); g.name = f'{a}^{power} mod 15'
        return g.control()

    def iqft(n):
        qc = QuantumCircuit(n)
        for q in range(n // 2): qc.swap(q, n - 1 - q)
        for j in range(n):
            for m in range(j): qc.cp(-np.pi / 2 ** (j - m), m, j)
            qc.h(j)
        g = qc.to_gate(); g.name = 'IQFT'
        return g

    n_count = 3
    qc = QuantumCircuit(QuantumRegister(n_count, 'count'), QuantumRegister(4, 'work'),
                        ClassicalRegister(n_count, 'meas'))
    for q in range(n_count): qc.h(q)
    qc.x(n_count)  # work register starts at |1>
    for q in range(n_count):
        qc.append(c_amod15(a, 2 ** q), [q] + list(range(n_count, n_count + 4)))
    qc.append(iqft(n_count), range(n_count))
    qc.measure(range(n_count), range(n_count))

    counts = run_circuit(qc, shots=1024)
    print('Measured phases (raw counts):', counts)

    factors = set()
    for bits in sorted(counts, key=counts.get, reverse=True):
        phase = int(bits, 2) / 2 ** n_count
        r = Fraction(phase).limit_denominator(N).denominator
        if r % 2 == 0:
            for f in (gcd(a ** (r // 2) - 1, N), gcd(a ** (r // 2) + 1, N)):
                if f not in (1, N): factors.add(f)
    if factors:
        print(f'Factors of {N} found: {sorted(factors)}')
    else:
        print('No non-trivial factor this time (noise/luck) - run again!')

In [ ]:
# ============ RUN AN EXAMPLE ============
# Un-comment ONE line and run this cell:

# example_1()   # Hello quantum: five 50/50 qubit measurements
# example_2()   # Bell state: entangled qubits
# example_3()   # Shor's algorithm: factor 15